# bounded-brain · LIBERO rollouts with privileged traces (π0.5)

This notebook spends spare Colab compute on data the project needs:

* It evaluates the open π0.5 LIBERO checkpoint with `lerobot-eval` (LeRobot 0.6.1), following the published
  protocol (10 episodes per task), so we can compare against reported success rates. The π0 checkpoint is
  probed too, but its weights do not load in LeRobot 0.6.1, so its runs are left out automatically.
* For **every** episode it also saves the simulator state at every step, object poses, gripper
  contacts and the task's goal predicates (`bb_trace.py`). These are the ground-truth labels the
  bounded-brain monitor will be evaluated on. Episodes can be replayed exactly offline (`bb_replay.py`).
* It adds out-of-distribution runs (LIBERO-90 tasks the checkpoints were not fine-tuned on), which
  produce failures, the interesting cases for a monitor.

**Before you start (about 5 minutes)**

1. *Runtime → Change runtime type → A100 GPU.* The A100 40 GB variant is cheaper than 80 GB; we do not need 80 GB.
2. π0 and π0.5 use the PaliGemma tokenizer, which is a **gated** Hugging Face repo. Log in to Hugging Face,
   open https://huggingface.co/google/paligemma-3b-pt-224 and accept the license (access is granted immediately).
   Then create a *read* token at https://huggingface.co/settings/tokens.
3. In Colab, open the key icon (**Secrets**) on the left, add `HF_TOKEN` with that token, and allow notebook access.
4. Run the cells in order. Keep this browser tab open while it runs.

**Two stages.** *Runtime → Run all* installs everything and runs the smoke tests (cell 6, about 20–30 min),
then stops at cell 7 on purpose. If cell 6 looks good, set `START_MAIN_RUNS = True` in cell 1 and fill in the two
numbers from Colab's Resources panel there, run cell 1, then run cells 7 and 8.

**When it stops.** The main runs stop by themselves before the compute budget or Colab's 12-hour session limit
runs out. Cell 8 then waits until every file has reached Google Drive (uploads lag behind the writes, and files
not yet uploaded are lost when the machine is released) and releases the GPU. If it stopped for the 12-hour
limit, open the notebook later and *Run all* again.

**Keep the browser connected.** Keep this tab open and the computer awake (no sleep), with a stable connection.
On 2026-10-17/18 a session ended after about 8 of its 12 hours; the cause is not known, and a lost browser
connection (sleep, network) is the most likely one.

**Disconnects.** Everything is written to Google Drive (`MyDrive/bounded-brain/libero_runs/`) as soon as each
episode ends. A short network drop does not stop the computation, which runs on Google's machine. If the runtime
itself is lost (idle timeout, a crash), run all cells again: finished units are skipped and only the interrupted
unit restarts; its partial episodes are kept separately and flagged. Before deleting a runtime by hand, run
`from google.colab import drive; drive.flush_and_unmount()` so the newest files are not lost.

In [ ]:
# @title 1 · Settings and GPU check
START_MAIN_RUNS = False      # set True once the smoke tests in cell 6 look good
# Budget for the main runs: two numbers from Colab's Resources panel (click the RAM/Disk bars at the top right,
# with this notebook connected to its A100). Changing either number later re-bases the budget; keeping them keeps it.
CU_AVAILABLE_NOW = None      # "Available: ... compute units"
CU_PER_HOUR_NOW = None       # "Usage rate: approximately ... per hour" (None = guess from the GPU type)
OUT_ROOT = "/content/drive/MyDrive/bounded-brain/libero_runs"
AUTO_DISCONNECT = True       # when the runs stop: wait for the upload to Drive, then release the GPU

import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none! (Runtime -> Change runtime type -> A100)")
# Third-party observation (March 2026): A100 40GB ~5.40 CU/h, A100 80GB ~7.52 CU/h, L4 ~1.71 CU/h.
CU_RATE = CU_PER_HOUR_NOW or (7.52 if "80GB" in gpu.replace(" ", "") else 5.40 if "A100" in gpu
                              else 1.71 if "L4" in gpu else 5.40)
print(f"Compute units: {CU_AVAILABLE_NOW} available, {CU_RATE} per hour"
      + ("" if CU_PER_HOUR_NOW else " (guessed from the GPU; the Resources panel shows the real rate)"))

In [ ]:
# @title 2 · Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(OUT_ROOT, exist_ok=True)
print("Output folder:", OUT_ROOT)

In [ ]:
# @title 3 · Hugging Face token and gated-model access
import os
from google.colab import userdata
from huggingface_hub import hf_hub_download

class SetupNeeded(Exception):
    pass

try:
    token = userdata.get("HF_TOKEN")
except Exception as exc:  # SecretNotFoundError, or NotebookAccessError if "Notebook access" is off
    raise SetupNeeded(
        f"{type(exc).__name__}: click the key icon on the left (Secrets) -> Add new secret, name HF_TOKEN, "
        "value = your Hugging Face *read* token (starts with hf_), and switch on 'Notebook access'. "
        "Then run this cell again.") from None
os.environ["HF_TOKEN"] = token
try:
    hf_hub_download("google/paligemma-3b-pt-224", "tokenizer_config.json", token=token)
    print("PaliGemma tokenizer access: OK")
except Exception as exc:
    raise SetupNeeded(
        f"{type(exc).__name__}: no access to google/paligemma-3b-pt-224. Open that model page on Hugging Face "
        "with the same account as the token, accept the license, then run this cell again.") from None

**4 · Install** LeRobot 0.6.1 and LIBERO in a separate Python 3.12 environment (about 5 minutes).

In [ ]:
%%bash
set -e
pip -q install uv
# Re-running this cell in the same runtime is fine: the environment is created only once.
[ -x /content/bbenv/bin/python ] || uv venv -q -p 3.12 /content/bbenv
# --torch-backend=auto picks the PyTorch CUDA build that matches this machine's driver.
uv pip install -q --python /content/bbenv/bin/python --torch-backend=auto "lerobot[libero,pi]==0.6.1"
/content/bbenv/bin/python -c "import torch, lerobot; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available(), '| lerobot', lerobot.__version__)"

In [ ]:
# @title 5 · Rendering (EGL) and LIBERO configuration
import os, subprocess, textwrap
icd = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"   # lets MuJoCo find NVIDIA's EGL driver
if not os.path.exists(icd):
    os.makedirs(os.path.dirname(icd), exist_ok=True)
    with open(icd, "w") as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')
setup = textwrap.dedent('''
    import os, yaml, importlib.util
    root = os.path.join(os.path.dirname(importlib.util.find_spec("libero").origin), "libero")
    paths = {"benchmark_root": "", "bddl_files": "bddl_files", "init_states": "init_files",
             "datasets": "../datasets", "assets": "assets"}
    cfg = {k: os.path.join(root, v) for k, v in paths.items()}
    os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)   # avoids LIBERO's interactive prompt
    with open(os.path.expanduser("~/.libero/config.yaml"), "w") as f:
        yaml.safe_dump(cfg, f)
    from libero.libero import get_assets_path
    print("LIBERO assets:", get_assets_path())      # downloads them once, before any parallel worker starts
    import mujoco
    ctx = mujoco.GLContext(64, 64); ctx.make_current(); ctx.free()
    print("MuJoCo rendering OK with", os.environ["MUJOCO_GL"])
''')

def try_backend(backend):
    env = dict(os.environ, MUJOCO_GL=backend, PYOPENGL_PLATFORM=backend, MPLBACKEND="Agg", PYTHONPATH="")
    r = subprocess.run(["/content/bbenv/bin/python", "-c", setup], env=env, capture_output=True, text=True)
    print(r.stdout[-1500:] or r.stderr[-1500:])
    return r.returncode == 0

MUJOCO_GL = "egl"
if not try_backend("egl"):
    print("EGL failed; falling back to OSMesa (CPU rendering, slower).")
    subprocess.run("apt-get -qq update && apt-get -qq install -y libosmesa6", shell=True, check=False)
    MUJOCO_GL = "osmesa"
    assert try_backend("osmesa"), "No working MuJoCo renderer; send me the output above."

The next cells write the trace-logging kit to `/content/bbkit` (the same files are in the
project repository under `experiments/colab/`).

In [ ]:
!mkdir -p /content/bbkit

In [ ]:
%%writefile /content/bbkit/bb_trace.py
"""bounded-brain trace logger for LeRobot LIBERO evaluations.

Importing this module monkeypatches ``lerobot.envs.libero.LiberoEnv`` so that every
episode run by ``lerobot-eval`` is also saved with privileged simulator information,
which the bounded-brain monitor needs as ground-truth labels:

* the MuJoCo simulator state at every step (enough to replay the episode exactly
  and recompute any predicate offline),
* end-effector pose and gripper joint positions,
* positions and orientations of every object and fixture,
* truth values of the task's BDDL goal predicates,
* per-object "grasped" and "gripper in contact" flags.

One ``<episode_id>.npz`` (arrays) and one ``<episode_id>.json`` (metadata) are written
per episode under ``$BB_TRACE_DIR/<suite>/task_<id>/``. Logging failures never stop
the evaluation; they are counted and reported in the metadata.

Written against lerobot==0.6.1 and hf-libero==0.1.4 (their source was read on
2026-10-17). Other versions may rename the internals used here; a warning is printed.
"""

from __future__ import annotations

import atexit
import gzip
import json
import os
import time
import uuid
import weakref
from importlib import metadata

import numpy as np

from lerobot.envs import libero as _libero

TESTED_VERSIONS = {"lerobot": "0.6.1", "hf-libero": "0.1.4"}
TRACE_DIR = os.environ.get("BB_TRACE_DIR", os.path.abspath("bb_traces"))
RUN_TAG = os.environ.get("BB_RUN_TAG", "run")
POLICY = os.environ.get("BB_POLICY", "unknown")
FORMAT_VERSION = 1


def _version(pkg: str) -> str:
    try:
        return metadata.version(pkg)
    except metadata.PackageNotFoundError:
        return "missing"


VERSIONS = {pkg: _version(pkg) for pkg in TESTED_VERSIONS}
for _pkg, _tested in TESTED_VERSIONS.items():
    if VERSIONS[_pkg] != _tested:
        print(f"[bb_trace] WARNING: {_pkg}=={VERSIONS[_pkg]} (tested with {_tested}); internals may differ.")

_LiberoEnv = _libero.LiberoEnv
_orig_ensure_env = _LiberoEnv._ensure_env
_orig_reset = _LiberoEnv.reset
_orig_step = _LiberoEnv.step
_orig_close = _LiberoEnv.close
_live = weakref.WeakSet()


def _suite_name(env) -> str:
    # The BDDL file lives in .../bddl_files/<suite>/<task>.bddl
    return os.path.basename(os.path.dirname(env._task_bddl_file))


def _wrap_capture(env, control_env) -> None:
    """Remember the last raw observation dict returned by the LIBERO control env."""

    def capture(fn):
        def wrapped(*args, **kwargs):
            out = fn(*args, **kwargs)
            raw = out[0] if isinstance(out, tuple) else out
            if isinstance(raw, dict):
                env._bb_raw = raw
            return out

        return wrapped

    for name in ("step", "reset", "set_init_state"):
        if hasattr(control_env, name):
            setattr(control_env, name, capture(getattr(control_env, name)))


def _ensure_env(self) -> None:
    fresh = self._env is None
    _orig_ensure_env(self)
    if fresh and self._env is not None:
        _wrap_capture(self, self._env)
        inner = self._env.env
        self._bb_objects = list(inner.objects_dict.keys())
        self._bb_fixtures = list(inner.fixtures_dict.keys())
        self._bb_goal = [list(map(str, g)) for g in inner.parsed_problem.get("goal_state", [])]
        _live.add(self)


def _note_error(self, where: str, exc: Exception) -> None:
    errs = self._bb_buf.setdefault("errors", {})
    if where not in errs:
        errs[where] = f"{type(exc).__name__}: {exc}"
    self._bb_buf["n_errors"] = self._bb_buf.get("n_errors", 0) + 1


def _vec(raw, key: str, n: int) -> np.ndarray:
    val = raw.get(key) if isinstance(raw, dict) else None
    return np.asarray(val, dtype=np.float32).reshape(n) if val is not None else np.full(n, np.nan, np.float32)


def _snapshot(self) -> dict:
    snap: dict = {}
    control = self._env
    inner = control.env
    try:
        snap["sim_state"] = np.asarray(control.get_sim_state(), dtype=np.float64).copy()
    except Exception as exc:  # noqa: BLE001
        _note_error(self, "sim_state", exc)
        snap["sim_state"] = None
    raw = getattr(self, "_bb_raw", None)
    snap["eef_pos"] = _vec(raw, "robot0_eef_pos", 3)
    snap["eef_quat"] = _vec(raw, "robot0_eef_quat", 4)
    snap["gripper_qpos"] = _vec(raw, "robot0_gripper_qpos", 2)
    names = self._bb_objects + self._bb_fixtures
    try:
        ids = [inner.obj_body_id[n] for n in names]
        snap["obj_pos"] = np.array(inner.sim.data.body_xpos[ids], dtype=np.float32)
        snap["obj_quat_wxyz"] = np.array(inner.sim.data.body_xquat[ids], dtype=np.float32)
    except Exception as exc:  # noqa: BLE001
        _note_error(self, "obj_pose", exc)
        snap["obj_pos"] = np.full((len(names), 3), np.nan, np.float32)
        snap["obj_quat_wxyz"] = np.full((len(names), 4), np.nan, np.float32)
    goal = []
    for g in self._bb_goal:
        try:
            goal.append(bool(inner._eval_predicate(g)))
        except Exception as exc:  # noqa: BLE001
            _note_error(self, "goal_predicate", exc)
            goal.append(False)
    snap["goal"] = np.array(goal, dtype=bool)
    grasp, contact = [], []
    try:
        gripper = inner.robots[0].gripper
    except Exception as exc:  # noqa: BLE001
        _note_error(self, "gripper", exc)
        gripper = None
    for n in self._bb_objects:
        g = c = False
        if gripper is not None:
            try:
                g = bool(inner._check_grasp(gripper=gripper, object_geoms=inner.objects_dict[n]))
            except Exception as exc:  # noqa: BLE001
                _note_error(self, "grasp", exc)
            try:
                c = bool(inner.check_contact(gripper, inner.objects_dict[n]))
            except Exception as exc:  # noqa: BLE001
                _note_error(self, "contact", exc)
        grasp.append(g)
        contact.append(c)
    snap["grasp"] = np.array(grasp, dtype=bool)
    snap["contact"] = np.array(contact, dtype=bool)
    return snap


def _start(self, seed, init_id) -> None:
    self._bb_buf = {
        "t0": time.time(),
        "seed": None if seed is None else int(seed),
        "init_state_id": init_id,
        "actions": [],
        "rewards": [],
        "success": [],
        "snaps": [],
        "model_xml_gz": None,
    }
    try:
        # Exact offline replay needs more than the sim state: the episode's model XML,
        # and the fixture placements that LIBERO writes into model.body_pos/body_quat
        # at reset time (bddl_base_domain.py), which the XML does not contain.
        inner = self._env.env
        self._bb_buf["model_xml_gz"] = gzip.compress(inner.model.get_xml().encode("utf-8"))
        self._bb_buf["model_body_pos"] = np.array(inner.sim.model.body_pos, dtype=np.float64)
        self._bb_buf["model_body_quat"] = np.array(inner.sim.model.body_quat, dtype=np.float64)
    except Exception as exc:  # noqa: BLE001
        _note_error(self, "model", exc)
    try:
        self._bb_buf["snaps"].append(_snapshot(self))
    except Exception as exc:  # noqa: BLE001
        _note_error(self, "snapshot", exc)


def _flush(self, reason: str) -> None:
    buf = getattr(self, "_bb_buf", None)
    self._bb_buf = None
    if not buf or not buf["actions"]:
        return
    try:
        suite = _suite_name(self)
        n = len(buf["actions"])
        succ = np.array(buf["success"], dtype=bool)
        init_id = buf["init_state_id"]
        ep_id = (
            f"{RUN_TAG}-{suite}-t{self.task_id:02d}-i{-1 if init_id is None else init_id:03d}"
            f"-s{buf['seed']}-{uuid.uuid4().hex[:8]}"
        )
        out_dir = os.path.join(TRACE_DIR, suite, f"task_{self.task_id:02d}")
        os.makedirs(out_dir, exist_ok=True)
        snaps = buf["snaps"]
        arrays = {
            "actions": np.asarray(buf["actions"], dtype=np.float32),
            "rewards": np.asarray(buf["rewards"], dtype=np.float32),
            "success": succ,
        }
        for key in ("eef_pos", "eef_quat", "gripper_qpos", "obj_pos", "obj_quat_wxyz", "goal", "grasp", "contact"):
            arrays[key] = np.stack([s[key] for s in snaps])
        if all(s.get("sim_state") is not None for s in snaps):
            arrays["sim_state"] = np.stack([s["sim_state"] for s in snaps])
        if buf.get("model_xml_gz") is not None:
            arrays["model_xml_gz"] = np.frombuffer(buf["model_xml_gz"], dtype=np.uint8)
        for key in ("model_body_pos", "model_body_quat"):
            if buf.get(key) is not None:
                arrays[key] = buf[key]
        np.savez_compressed(os.path.join(out_dir, ep_id + ".npz"), **arrays)
        meta = {
            "format_version": FORMAT_VERSION,
            "episode_id": ep_id,
            "run_tag": RUN_TAG,
            "policy": POLICY,
            "suite": suite,
            "task_id": int(self.task_id),
            "task_name": self.task,
            "language": self.task_description,
            "init_state_id": init_id,
            "seed": buf["seed"],
            "n_steps": n,
            "n_snapshots": len(snaps),
            "success": bool(succ.any()),
            "success_step": int(np.argmax(succ)) if succ.any() else None,
            "end_reason": reason,
            "max_episode_steps": int(self._max_episode_steps),
            "control_freq_hz": int(self.control_freq),
            "objects": self._bb_objects,
            "fixtures": self._bb_fixtures,
            "goal_state": self._bb_goal,
            "array_shapes": {k: list(v.shape) for k, v in arrays.items()},
            "wall_time_s": round(time.time() - buf["t0"], 2),
            "versions": VERSIONS,
            "logger_errors": buf.get("errors", {}),
            "n_logger_errors": buf.get("n_errors", 0),
        }
        with open(os.path.join(out_dir, ep_id + ".json"), "w") as f:
            json.dump(meta, f, indent=1)
    except Exception as exc:  # noqa: BLE001
        print(f"[bb_trace] WARNING: could not save an episode ({type(exc).__name__}: {exc})")


def reset(self, seed=None, **kwargs):
    _flush(self, "reset")
    init_id = None
    if self.init_states and self._init_states is not None:
        init_id = int(self.init_state_id % len(self._init_states))
    self._bb_raw = None
    out = _orig_reset(self, seed=seed, **kwargs)
    _start(self, seed, init_id)
    return out


def step(self, action):
    out = _orig_step(self, action)
    _obs, reward, terminated, truncated, info = out
    buf = getattr(self, "_bb_buf", None)
    if buf is not None:
        try:
            buf["actions"].append(np.asarray(action, dtype=np.float32).reshape(-1))
            buf["rewards"].append(float(reward))
            buf["success"].append(bool(info.get("is_success", False)))
            buf["snaps"].append(_snapshot(self))
        except Exception as exc:  # noqa: BLE001
            _note_error(self, "step", exc)
        if terminated or truncated:
            _flush(self, "success" if info.get("is_success", False) else "terminated")
        elif len(buf["actions"]) >= self._max_episode_steps:
            _flush(self, "max_steps")
    return out


def close(self):
    _flush(self, "close")
    return _orig_close(self)


@atexit.register
def _flush_all() -> None:
    for env in list(_live):
        _flush(env, "exit")


_LiberoEnv._ensure_env = _ensure_env
_LiberoEnv.reset = reset
_LiberoEnv.step = step
_LiberoEnv.close = close
print(f"[bb_trace] logging LIBERO episodes to {TRACE_DIR} (run={RUN_TAG}, policy={POLICY})")

In [ ]:
%%writefile /content/bbkit/bb_eval.py
"""Run `lerobot-eval` with the bounded-brain trace logger switched on.

Usage (same flags as lerobot-eval):
    BB_TRACE_DIR=/path/to/traces BB_RUN_TAG=pi05_spatial BB_POLICY=lerobot/pi05_libero_finetuned \
        python bb_eval.py --policy.path=lerobot/pi05_libero_finetuned --env.type=libero ...

BB_MAX_VIDEOS (default 1) caps how many episodes per task lerobot-eval renders to mp4.
lerobot 0.6.1 renders up to 10 per task, which re-renders a frame every step; the
traces already allow exact offline replay, so a single video per task is enough.
"""

import os
import sys

import bb_trace  # noqa: F401  (patches LiberoEnv before any env is created)
import lerobot.scripts.lerobot_eval as _le

_MAX_VIDEOS = int(os.environ.get("BB_MAX_VIDEOS", "1"))
_orig_eval_policy = _le.eval_policy


def _eval_policy(*args, **kwargs):
    if "max_episodes_rendered" in kwargs:
        kwargs["max_episodes_rendered"] = min(int(kwargs["max_episodes_rendered"]), _MAX_VIDEOS)
    return _orig_eval_policy(*args, **kwargs)


_le.eval_policy = _eval_policy

if __name__ == "__main__":
    sys.argv[0] = "lerobot-eval"
    _le.main()

In [ ]:
%%writefile /content/bbkit/bb_replay.py
"""Replay a logged LIBERO episode exactly (offline, CPU is enough).

    from bb_replay import Replayer
    rep = Replayer("path/to/<episode_id>.npz")   # reads the matching .json too
    for k in range(rep.n_snapshots):
        inner = rep.goto(k)                       # robosuite/LIBERO env at snapshot k
        on = inner._eval_predicate(["on", "akita_black_bowl_1", "plate_1"])

Any predicate, contact or distance can be recomputed this way, so labels for new
specifications do not require re-running the policy.
"""

from __future__ import annotations

import gzip
import json
import os

import numpy as np


class Replayer:
    def __init__(self, npz_path: str):
        from libero.libero import benchmark
        from lerobot.envs.libero import LiberoEnv

        self.data = np.load(npz_path)
        with open(os.path.splitext(npz_path)[0] + ".json") as f:
            self.meta = json.load(f)
        if "sim_state" not in self.data.files or "model_xml_gz" not in self.data.files:
            raise ValueError("this trace has no sim_state/model_xml_gz; exact replay is impossible")
        suite = benchmark.get_benchmark_dict()[self.meta["suite"]]()
        self.env = LiberoEnv(
            task_suite=suite,
            task_id=self.meta["task_id"],
            task_suite_name=self.meta["suite"],
            obs_type="pixels_agent_pos",
        )
        self.env._ensure_env()
        xml = gzip.decompress(self.data["model_xml_gz"].tobytes()).decode("utf-8")
        self.env._env.reset_from_xml_string(xml)
        model = self.env._env.env.sim.model
        # LIBERO places fixtures by writing model.body_pos/body_quat at reset time.
        if "model_body_pos" in self.data.files:
            model.body_pos[:] = self.data["model_body_pos"]
            model.body_quat[:] = self.data["model_body_quat"]
        self.n_snapshots = int(self.data["sim_state"].shape[0])

    def goto(self, k: int):
        control = self.env._env
        control.set_state(self.data["sim_state"][k])
        control.env.sim.forward()
        return control.env

    def close(self) -> None:
        self.env.close()

In [ ]:
%%writefile /content/bbkit/bb_colab.py
"""Helpers for the bounded-brain Colab notebook.

Runs in Colab's own Python. Each evaluation "unit" (one policy x one LIBERO suite x one
seed) is a separate `lerobot-eval` process inside the /content/bbenv virtualenv, with the
trace logger switched on. Units are resumable: a unit whose eval_info.json exists on
Google Drive is skipped, so after a disconnect you just run the notebook again.
"""

from __future__ import annotations

import glob
import json
import os
import re
import shutil
import signal
import subprocess
import threading
import time

VENV_PY = "/content/bbenv/bin/python"
KIT = "/content/bbkit"
SUITE_TASKS = {"libero_spatial": 10, "libero_object": 10, "libero_goal": 10, "libero_10": 10, "libero_90": 90}
ECHO_KEYS = ("pc_success", "Traceback", "Error", "error:", "bb_trace", "Killed", "CUDA out of memory",
             "Could not load state dict", "Missing keys", "Unexpected keys")


def _ledger_path(out_root: str) -> str:
    return os.path.join(out_root, "ledger.json")


def load_ledger(out_root: str) -> dict:
    try:
        with open(_ledger_path(out_root)) as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return {"units": {}, "gpu_seconds": 0.0}


def save_ledger(out_root: str, ledger: dict) -> None:
    os.makedirs(out_root, exist_ok=True)
    tmp = _ledger_path(out_root) + ".tmp"
    with open(tmp, "w") as f:
        json.dump(ledger, f, indent=1)
    os.replace(tmp, _ledger_path(out_root))


def count_traces(trace_dir: str) -> int:
    return len(glob.glob(os.path.join(trace_dir, "**", "*.json"), recursive=True))


# lerobot 0.6.1's pi0/pi05 `from_pretrained` catches a failed `load_state_dict`, prints one of these
# lines and carries on with partly random weights. Such a unit must not count as valid data.
WEIGHT_WARNINGS = ("Could not load state dict", "Missing keys when loading state dict",
                   "Unexpected keys when loading state dict", "Returning model without loading pretrained weights")


def weight_warning(log_path: str) -> str | None:
    """First weight-loading warning in a unit's log, or None."""
    try:
        with open(log_path, errors="replace") as f:
            for line in f:
                if any(w in line for w in WEIGHT_WARNINGS):
                    return line.strip()[:300]
    except OSError:
        pass
    return None


def _summary_from_disk(unit: str, eval_dir: str, unit_trace_dir: str, log_path: str, previous: dict) -> dict:
    """Rebuild a finished unit's summary from its files (the ledger entry may be stale)."""
    summary = dict(previous or {})
    summary.update({"unit": unit, "status": "done", "episodes_logged": count_traces(unit_trace_dir), "log": log_path})
    try:
        with open(os.path.join(eval_dir, "eval_info.json")) as f:
            overall = json.load(f).get("overall", {})
        summary["pc_success"] = overall.get("pc_success")
        summary["n_episodes"] = overall.get("n_episodes")
    except (OSError, json.JSONDecodeError):
        pass
    warn = weight_warning(log_path)
    if warn:
        summary["status"] = "failed"
        summary["weights_warning"] = warn
    return summary


def run_unit(
    out_root: str,
    tag: str,
    policy: str,
    suite: str,
    n_episodes: int,
    seed: int,
    batch_size: int,
    use_async: bool,
    extra: tuple = (),
    task_ids: list | None = None,
    episode_length: int | None = None,
    max_videos: int = 1,
    mujoco_gl: str = "egl",
    heartbeat_s: int = 300,
    timeout_s: float | None = None,
) -> dict:
    """Run one lerobot-eval process; return a summary dict (also stored in the ledger).

    timeout_s: kill the whole process group (including parallel env workers) after this
    many seconds and mark the unit as failed, so a hung run cannot block the notebook.
    Default: 20 min + 30 s per episode (60 s without parallel envs), at least 30 min.
    """
    if timeout_s is None:
        n_tasks = len(task_ids) if task_ids is not None else SUITE_TASKS.get(suite, 10)
        timeout_s = max(1800.0, 1200.0 + 30.0 * n_tasks * n_episodes * (1 if use_async else 2))
    unit = f"{tag}/{suite}"
    eval_dir = os.path.join(out_root, "evals", tag, suite)
    trace_dir = os.path.join(out_root, "traces", tag)
    unit_trace_dir = os.path.join(trace_dir, suite)  # bb_trace writes <trace_dir>/<suite>/task_XX/
    log_dir = os.path.join(out_root, "logs")
    os.makedirs(log_dir, exist_ok=True)
    info_path = os.path.join(eval_dir, "eval_info.json")
    log_path = os.path.join(log_dir, f"{tag}__{suite}.log")
    ledger = load_ledger(out_root)
    if os.path.exists(info_path):
        summary = _summary_from_disk(unit, eval_dir, unit_trace_dir, log_path, ledger["units"].get(unit))
        if ledger["units"].get(unit) != summary:
            ledger["units"][unit] = summary
            save_ledger(out_root, ledger)
        note = f" (weights did not load: {summary['weights_warning']})" if summary.get("weights_warning") else ""
        print(f"[skip] {unit}: already run{note}")
        return summary
    # An earlier attempt was cut off (runtime lost, timeout, error). Keep its data, but apart from
    # the new attempt, so finished units never contain duplicate episodes.
    stamp = int(time.time())
    for d in (eval_dir, unit_trace_dir):
        if os.path.exists(d):
            shutil.move(d, f"{d}__partial_{stamp}")
            print(f"[keep] earlier partial attempt kept at {d}__partial_{stamp}")

    cmd = [
        VENV_PY, os.path.join(KIT, "bb_eval.py"),
        f"--policy.path={policy}",
        "--env.type=libero",
        f"--env.task={suite}",
        f"--eval.n_episodes={n_episodes}",
        f"--eval.batch_size={batch_size}",
        f"--eval.use_async_envs={'true' if use_async else 'false'}",
        f"--seed={seed}",
        f"--output_dir={eval_dir}",
        *extra,
    ]
    if task_ids is not None:
        cmd.append("--env.task_ids=" + json.dumps(list(task_ids)).replace(" ", ""))
    if episode_length is not None:
        cmd.append(f"--env.episode_length={episode_length}")
    # The eval runs in a separate Python 3.12 venv; do not leak Colab kernel settings into it.
    # Colab sets MPLBACKEND to its inline backend, which the venv lacks (matplotlib then fails on import),
    # and PYTHONPATH to its own 3.13 packages.
    env = dict(
        os.environ,
        MPLBACKEND="Agg",
        MUJOCO_GL=mujoco_gl,
        PYOPENGL_PLATFORM=mujoco_gl,
        BB_TRACE_DIR=trace_dir,
        BB_RUN_TAG=tag,
        BB_POLICY=policy,
        BB_MAX_VIDEOS=str(max_videos),
        PYTHONPATH=KIT,
        TQDM_DISABLE="1",
        TOKENIZERS_PARALLELISM="false",
    )
    print(f"[run ] {unit}: {policy} | {n_episodes} ep/task | batch {batch_size} | "
          f"{'async' if use_async else 'sync'} | seed {seed}")
    t0 = time.time()
    stop = threading.Event()
    lock = threading.Lock()
    credited = [0.0]

    def credit(seconds: float, final: dict | None = None) -> None:
        # GPU time is booked every heartbeat, so it is not lost if the runtime dies mid-unit.
        with lock:
            led = load_ledger(out_root)
            led["gpu_seconds"] = led.get("gpu_seconds", 0.0) + seconds
            led["units"][unit] = final or {"unit": unit, "status": "running",
                                           "minutes": round((time.time() - t0) / 60, 2),
                                           "episodes_logged": count_traces(unit_trace_dir)}
            save_ledger(out_root, led)
            credited[0] += seconds

    def heartbeat():
        while not stop.wait(heartbeat_s):
            credit(heartbeat_s)
            print(f"       ... {unit}: {count_traces(unit_trace_dir)} episodes logged, "
                  f"{(time.time() - t0) / 60:.1f} min")

    # An eval left over from an interrupted cell would still hold the GPU: stop it first. The anchored
    # pattern only matches command lines that start with the venv's python running our bb_eval.py.
    subprocess.run(["pkill", "-f", "^" + re.escape(VENV_PY) + " " + re.escape(os.path.join(KIT, "bb_eval.py"))],
                   check=False)
    hb = threading.Thread(target=heartbeat, daemon=True)
    hb.start()
    timed_out = threading.Event()

    def kill_group(proc):
        try:
            os.killpg(proc.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass

    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,
                                start_new_session=True)

        def watchdog():
            if not stop.wait(timeout_s) and proc.poll() is None:
                timed_out.set()
                print(f"       !!! {unit}: no result after {timeout_s / 60:.0f} min, stopping it")
                kill_group(proc)

        threading.Thread(target=watchdog, daemon=True).start()
        banner_shown = False
        try:
            for line in proc.stdout:
                log.write(line)
                if "[bb_trace] logging LIBERO episodes" in line:   # printed by every parallel env worker
                    if banner_shown:
                        continue
                    banner_shown = True
                if any(k in line for k in ECHO_KEYS):
                    print("       " + line.rstrip()[:220])
            rc = proc.wait()
        except BaseException:
            # The cell was interrupted: the eval runs in its own process group, so stop it explicitly.
            stop.set()
            kill_group(proc)
            raise
    stop.set()
    hb.join(timeout=60)
    if timed_out.is_set():
        rc = rc if rc not in (0, None) else -9
    elapsed = time.time() - t0
    n_eps = count_traces(unit_trace_dir)
    summary = {
        "unit": unit, "policy": policy, "suite": suite, "seed": seed, "batch_size": batch_size,
        "async": use_async, "returncode": rc, "minutes": round(elapsed / 60, 2),
        "episodes_logged": n_eps, "status": "done" if (rc == 0 and os.path.exists(info_path)) else "failed",
        "timed_out": timed_out.is_set(), "log": log_path,
    }
    if summary["status"] == "done":
        with open(info_path) as f:
            info = json.load(f)
        overall = info.get("overall", {})
        summary["pc_success"] = overall.get("pc_success")
        summary["n_episodes"] = overall.get("n_episodes")
        summary["sec_per_episode"] = round(elapsed / max(1, n_eps), 2)
    warn = weight_warning(log_path)
    if warn:  # the policy ran with partly random weights: its episodes are not valid data
        summary["status"] = "failed"
        summary["weights_warning"] = warn
    credit(max(0.0, elapsed - credited[0]), final=summary)
    status = summary["status"].upper()
    print(f"[{status}] {unit}: {summary.get('pc_success')}% success, {n_eps} episodes, "
          f"{summary['minutes']} min (log: {log_path})" + (f"\n       WEIGHTS DID NOT LOAD: {warn}" if warn else ""))
    return summary


def episode_index(out_root: str, run_tag: str = "") -> list[dict]:
    """One row per logged episode (all runs, or only `run_tag`'s, which reads far fewer files)."""
    rows = []
    for j in glob.glob(os.path.join(out_root, "traces", run_tag, "**", "*.json"), recursive=True):
        try:
            with open(j) as f:
                m = json.load(f)
        except (OSError, json.JSONDecodeError):
            continue
        row = {k: m.get(k) for k in (
            "episode_id", "run_tag", "policy", "suite", "task_id", "language", "init_state_id", "seed",
            "n_steps", "success", "success_step", "end_reason", "n_logger_errors")}
        row["partial_attempt"] = "__partial_" in j  # from a unit attempt that was cut off
        row["path"] = os.path.relpath(j, out_root)
        rows.append(row)
    return rows


def export_zip(out_root: str, dest_name: str = "bb_traces_export.zip") -> str:
    """Pack traces, eval summaries, logs, ledger and index into one zip on Drive (no videos)."""
    import zipfile

    local = os.path.join("/content", dest_name) if os.path.isdir("/content") else os.path.join(out_root, "_" + dest_name)
    patterns = ["traces/**/*.npz", "traces/**/*.json", "evals/**/eval_info.json", "logs/*.log",
                "ledger.json", "episodes.csv"]
    n = 0
    with zipfile.ZipFile(local, "w", compression=zipfile.ZIP_STORED) as z:  # .npz is already compressed
        for pat in patterns:
            for path in glob.glob(os.path.join(out_root, pat), recursive=True):
                if os.path.isfile(path):
                    z.write(path, os.path.relpath(path, out_root))
                    n += 1
    dest = os.path.join(out_root, dest_name)
    if os.path.abspath(local) != os.path.abspath(dest):
        shutil.copy(local, dest)
    print(f"{n} files, {os.path.getsize(dest) / 1e6:.0f} MB -> {dest}")
    return dest

In [ ]:
# @title 6 · Smoke tests (about 10–20 min): π0.5 sync, π0.5 parallel envs, π0
import sys, importlib, json
sys.path.insert(0, "/content/bbkit")
import bb_colab; importlib.reload(bb_colab)

# n_action_steps=10 follows LeRobot's LIBERO docs. The checkpoints switch on torch.compile in max-autotune mode,
# which spends many minutes autotuning in every new process; inference is a small share of an episode, so turn it off.
PI05, PI05_ARGS = "lerobot/pi05_libero_finetuned", ("--policy.n_action_steps=10", "--policy.compile_model=false")
PI0, PI0_ARGS = "lerobot/pi0_libero", ("--policy.compile_model=false",)

def probe(tag, skip_if_failed=False, **kw):
    # Only the optional parallel-env probe is not retried; the required smoke tests always run again.
    prev = bb_colab.load_ledger(OUT_ROOT)["units"].get(f"{tag}/{kw['suite']}")
    if skip_if_failed and prev and prev.get("status") == "failed":
        print(f"[skip] {tag}: failed in an earlier session"); return prev
    return bb_colab.run_unit(OUT_ROOT, tag, mujoco_gl=MUJOCO_GL, heartbeat_s=120, **kw)

s1 = probe("smoke_pi05_sync", policy=PI05, suite="libero_spatial", n_episodes=1, seed=0,   # includes the model download
           batch_size=1, use_async=False, extra=PI05_ARGS, task_ids=[0], timeout_s=2400)
assert s1.get("status") == "done", "The basic smoke test failed. Send me the last 40 lines of the log file printed above."
for r in bb_colab.episode_index(OUT_ROOT, run_tag="smoke_pi05_sync"):
    print("trace:", r)

s2 = probe("smoke_pi05_async", skip_if_failed=True, policy=PI05, suite="libero_object", n_episodes=2, seed=0,
           batch_size=2, use_async=True, extra=PI05_ARGS, task_ids=[0], timeout_s=900)
ASYNC_OK = s2.get("status") == "done" and s2.get("episodes_logged", 0) >= 2
s3 = probe("smoke_pi0_sync", policy=PI0, suite="libero_spatial", n_episodes=1, seed=0,     # includes the model download
           batch_size=1, use_async=False, extra=PI0_ARGS, task_ids=[0], timeout_s=2400)
PI0_OK = s3.get("status") == "done"     # "failed" also when its weights did not load (then the runs are skipped)
print(f"\nparallel envs work: {ASYNC_OK} | pi0 checkpoint usable: {PI0_OK} | "
      f"pi0.5 seconds per episode (single env, incl. loading): {s1.get('sec_per_episode')}")
if s3.get("weights_warning"):
    print("pi0: the checkpoint's weights do not load in LeRobot 0.6.1, so pi0 runs are left out of the plan.")
print("\nSMOKE TESTS DONE. Copy this cell's output to Claude. To start the main runs: in cell 1 set "
      "START_MAIN_RUNS = True and fill in CU_AVAILABLE_NOW / CU_PER_HOUR_NOW, run cell 1, then run cells 7 and 8.")

In [ ]:
# @title 7 · Main runs (they stop by themselves before the budget or Colab's 12-hour limit runs out)
SESSION_LIMIT_H = 12.0    # Colab Pro / pay-as-you-go sessions end after at most 12 hours (Colab FAQ)
END_RESERVE_H = 0.5       # kept free at the end of a session for the summary and the upload to Drive
FINAL_RESERVE_H = 0.75    # kept out of the budget: the final summary and the setup of one more session
STALL_MIN = 25            # no new episode for this long (45 min before a unit's first one): the run is hung, stop it
FIRST_EPISODE_MIN = 45

import glob, os, re, shutil, signal, subprocess, threading, time, traceback

class MainRunsNotStarted(Exception):
    pass

if not START_MAIN_RUNS:
    raise MainRunsNotStarted("Stopped before the main runs on purpose. If the smoke tests look good: in cell 1 set "
                             "START_MAIN_RUNS = True and fill in CU_AVAILABLE_NOW / CU_PER_HOUR_NOW, run cell 1, "
                             "then run this cell and cell 8.")

# Session clock: the kernel's start time, kept in a file on this machine so re-running cells does not reset it.
_mark = "/content/.bb_session_start"
if not os.path.exists(_mark):
    try:
        import psutil
        _t0 = psutil.Process(os.getpid()).create_time()
    except Exception:
        _t0 = time.time()
    with open(_mark, "w") as f:
        f.write(str(_t0))
with open(_mark) as f:
    SESSION_START = float(f.read())

def session_hours():
    return (time.time() - SESSION_START) / 3600

# Budget in hours of runs, from cell 1's numbers. It is re-based only when those numbers change, so a restart that
# keeps them continues the same budget.
ledger = bb_colab.load_ledger(OUT_ROOT)
panel = [CU_AVAILABLE_NOW, CU_PER_HOUR_NOW]
if CU_AVAILABLE_NOW is not None and ledger.get("budget_from_panel") != panel:
    ledger["budget_hours"] = max(0.0, CU_AVAILABLE_NOW / CU_RATE - FINAL_RESERVE_H)
    ledger["gpu_seconds_at_budget_start"] = ledger.get("gpu_seconds", 0.0)
    ledger["budget_from_panel"] = panel
    bb_colab.save_ledger(OUT_ROOT, ledger)
    print(f"Budget set from cell 1: {CU_AVAILABLE_NOW} CU at {CU_RATE} CU/h.")
elif "budget_hours" not in ledger:
    raise MainRunsNotStarted("Fill in CU_AVAILABLE_NOW in cell 1 (Colab's Resources panel), run cell 1, "
                             "then run this cell and cell 8.")
budget_h = ledger["budget_hours"]
start_s = ledger.get("gpu_seconds_at_budget_start", 0.0)

def used_hours():
    return (bb_colab.load_ledger(OUT_ROOT)["gpu_seconds"] - start_s) / 3600

MAX_STEPS = {"libero_spatial": 280, "libero_object": 280, "libero_goal": 300, "libero_10": 520, "libero_90": 400}

def unit_hours(suite):
    # Expected duration of a unit, scaled from the main units finished so far (same suite if there is one).
    done = [s for u, s in bb_colab.load_ledger(OUT_ROOT)["units"].items()
            if not u.startswith("smoke_") and s.get("status") == "done" and s.get("minutes") and s.get("suite") in MAX_STEPS]
    same = [s for s in done if s["suite"] == suite]
    pool, factor = (same, 1.15) if same else (done, 1.5)
    work = bb_colab.SUITE_TASKS[suite] * MAX_STEPS[suite]          # tasks x step limit
    if not pool:
        return work * 0.25 / 3600                                   # before any unit has finished
    per_work = (sum(s["minutes"] * 60 for s in pool)
                / sum(bb_colab.SUITE_TASKS[s["suite"]] * MAX_STEPS[s["suite"]] for s in pool))
    return factor * work * per_work / 3600

def timeout_for(suite, n_ep, need_h, use_async):
    # Hung runs are caught by the progress watch below, so the hard limit is generous: 3x the expected time.
    default = max(1800.0, 1200.0 + 30.0 * bb_colab.SUITE_TASKS[suite] * n_ep * (1 if use_async else 2))
    return max(default, 3 * need_h * 3600)

def kill_evals():
    # Same anchored pattern as bb_colab: only our own lerobot-eval processes; their whole process group.
    pattern = "^" + re.escape(bb_colab.VENV_PY) + " " + re.escape(os.path.join(bb_colab.KIT, "bb_eval.py"))
    for pid in subprocess.run(["pgrep", "-f", pattern], capture_output=True, text=True).stdout.split():
        try:
            os.killpg(os.getpgid(int(pid)), signal.SIGKILL)
        except (ProcessLookupError, PermissionError, ValueError):
            pass

def watch_progress(unit_trace_dir, done, poll_s=60):
    last_n, last_t = None, time.time()
    while not done.wait(poll_s):
        n = bb_colab.count_traces(unit_trace_dir)
        if n != last_n:
            last_n, last_t = n, time.time()
        elif time.time() - last_t > (STALL_MIN if n else FIRST_EPISODE_MIN) * 60:
            print(f"       !!! no new episode for {(time.time() - last_t) / 60:.0f} min: stopping this unit as hung")
            kill_evals()
            return

def run_watched(tag, use_async, need_h, **kw):
    done = threading.Event()
    threading.Thread(target=watch_progress, args=(os.path.join(OUT_ROOT, "traces", tag, kw["suite"]), done),
                     daemon=True).start()
    try:
        return bb_colab.run_unit(OUT_ROOT, tag, use_async=use_async,
                                 timeout_s=timeout_for(kw["suite"], kw["n_episodes"], need_h, use_async), **kw)
    finally:
        done.set()

def blocker(need_h):
    if used_hours() + need_h > budget_h:
        return "budget"
    if session_hours() + need_h + END_RESERVE_H > SESSION_LIMIT_H:
        return "session_limit"
    return None

print(f"Budget {budget_h:.1f} h for the main runs, used so far {used_hours():.1f} h. "
      f"This session has run {session_hours():.1f} h of at most {SESSION_LIMIT_H:.0f} h.")

STD = ["libero_spatial", "libero_object", "libero_goal", "libero_10"]
RANDOM_INIT = ("--env.init_states=false",)   # random object layouts instead of LIBERO's 50 fixed ones
plan  = [("pi05_s1000", PI05, s, 10, 1000, PI05_ARGS) for s in STD]                        # 1 reproduce the published protocol
plan += [("pi05_ood90_s1000", PI05, "libero_90", 3, 1000, PI05_ARGS)]                      # 2 unseen tasks -> failures
if PI0_OK:
    plan += [("pi0_s1000", PI0, s, 10, 1000, PI0_ARGS) for s in STD]                        # 3 a second, weaker policy
    plan += [("pi0_ood90_s1000", PI0, "libero_90", 3, 1000, PI0_ARGS)]                      # 4
plan += [("pi05_randinit_s1000", PI05, s, 5, 1000, PI05_ARGS + RANDOM_INIT) for s in STD]    # 5 new layouts
plan += [("pi05_ood90_randinit_s2000", PI05, "libero_90", 3, 2000, PI05_ARGS + RANDOM_INIT)] # 6 more, different failures
if PI0_OK:
    plan += [("pi0_randinit_s1000", PI0, s, 5, 1000, PI0_ARGS + RANDOM_INIT) for s in STD]  # 7
plan += [("pi05_s2000", PI05, s, 10, 2000, PI05_ARGS) for s in STD]                        # 8 extra seeds (for mean ± sd
plan += [("pi05_s3000", PI05, s, 10, 3000, PI05_ARGS) for s in STD]                        #   of the reproduction)
for seed in range(4000, 40000, 1000):                                                     # 9 then new layouts until the
    plan += [(f"pi05_ood90_randinit_s{seed}", PI05, "libero_90", 3, seed, PI05_ARGS + RANDOM_INIT)]   # budget is used
    plan += [(f"pi05_randinit_s{seed}", PI05, s, 5, seed, PI05_ARGS + RANDOM_INIT) for s in STD]

def finished(tag, suite):
    return os.path.exists(os.path.join(OUT_ROOT, "evals", tag, suite, "eval_info.json"))

# Units that are valued for their individual episodes rather than as a complete set (random layouts, unseen tasks).
# If such a unit was cut off after logging at least half of its episodes, it is not run again: the same seed gives
# the same layouts, so a re-run would mostly repeat those episodes. They are kept, flagged `partial_attempt`.
EXPLORATORY = ("randinit", "ood90")

def set_aside(tag, suite):
    # Move a cut-off attempt's folders aside (as run_unit would) and count the episodes all such attempts logged.
    stamp = int(time.time())
    for d in (os.path.join(OUT_ROOT, "evals", tag, suite), os.path.join(OUT_ROOT, "traces", tag, suite)):
        if os.path.exists(d):
            shutil.move(d, f"{d}__partial_{stamp}")
            print(f"[keep] earlier partial attempt kept at {d}__partial_{stamp}")
    return sum(bb_colab.count_traces(d)
               for d in glob.glob(os.path.join(OUT_ROOT, "traces", tag, glob.escape(suite) + "__partial_*")))

RUN_STATE = "finished"     # read by cell 8: finished | budget | session_limit | error
skipped_for_budget, failures_in_a_row = [], 0
try:
    for tag, policy, suite, n_ep, seed, extra in plan:
        kw = dict(policy=policy, suite=suite, n_episodes=n_ep, seed=seed, batch_size=n_ep, extra=extra, mujoco_gl=MUJOCO_GL)
        if not finished(tag, suite) and any(k in tag for k in EXPLORATORY):
            n_all, n_cut = n_ep * bb_colab.SUITE_TASKS[suite], set_aside(tag, suite)
            if n_cut >= n_all / 2:
                print(f"[skip] {tag}/{suite}: a cut-off attempt already logged {n_cut} of {n_all} episodes "
                      "(kept, flagged partial)")
                continue
        need_h = unit_hours(suite)
        why = None if finished(tag, suite) else blocker(need_h)
        if why == "budget":
            skipped_for_budget.append(f"{tag}/{suite}")
            continue                     # a smaller unit further down may still fit
        if why == "session_limit":       # it fits the budget, but not the rest of this session
            RUN_STATE = "session_limit"
            print(f"[stop] {tag}/{suite} needs about {need_h:.1f} h and would not finish before Colab's "
                  f"{SESSION_LIMIT_H:.0f}-hour limit (this session has run {session_hours():.1f} h).")
            break
        s = run_watched(tag, ASYNC_OK, need_h, **kw)    # units finished earlier are only re-checked
        if s.get("status") == "failed" and ASYNC_OK and blocker(2 * need_h) is None:
            print("Retrying this unit with a single process.")
            s = run_watched(tag, False, 2 * need_h, **kw)
        failures_in_a_row = failures_in_a_row + 1 if s.get("status") == "failed" else 0
        if failures_in_a_row >= 3:
            RUN_STATE = "error"
            print("[stop] three units in a row failed, so the runs stop here instead of using up the budget. "
                  "Send Claude this cell's output.")
            break
except Exception:  # not KeyboardInterrupt: the stop button still just stops
    RUN_STATE = "error"
    traceback.print_exc()
    print("[stop] unexpected error (above). Cell 8 still uploads everything and releases the GPU.")
if skipped_for_budget:
    RUN_STATE = "budget" if RUN_STATE == "finished" else RUN_STATE
    print(f"[budget] {len(skipped_for_budget)} planned units did not fit in the remaining budget "
          f"(first: {skipped_for_budget[0]}).")
print(f"Main runs stopped: {RUN_STATE}. Budget used {used_hours():.1f} of {budget_h:.1f} h; "
      f"session at {session_hours():.1f} h.")

In [ ]:
# @title 8 · Summary, wait for the upload to Drive, release the GPU
import csv, collections, os, traceback
from google.colab import drive
RUN_STATE = globals().get("RUN_STATE", "finished")
try:
    ledger = bb_colab.load_ledger(OUT_ROOT)
    print(f"GPU time used by evaluations: {ledger['gpu_seconds'] / 3600:.2f} h\n")
    print(f"{'unit':40s} {'status':7s} {'success %':>9s} {'episodes':>8s} {'min':>6s}")
    for unit, s in sorted(ledger["units"].items()):
        print(f"{unit:40s} {s.get('status', '?'):7s} {str(s.get('pc_success')):>9s} "
              f"{s.get('episodes_logged') or 0:8d} {s.get('minutes') or 0:6.1f}")
    if RUN_STATE != "session_limit":   # the episode index and the zip are made once, at the very end
        rows = bb_colab.episode_index(OUT_ROOT)
        with open(os.path.join(OUT_ROOT, "episodes.csv"), "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else ["episode_id"])
            w.writeheader(); w.writerows(rows)
        complete = [r for r in rows if not r["partial_attempt"]]
        by = collections.Counter((r["run_tag"], r["success"]) for r in complete)
        print(f"\n{len(complete)} episodes with traces ({len(rows) - len(complete)} more from cut-off attempts); "
              "failures per run:", {tag: n for (tag, ok), n in sorted(by.items()) if not ok})
        print("Packing traces for download (no videos) ...")
        bb_colab.export_zip(OUT_ROOT)      # download this one file from Drive and put it in bounded-brain/data/
except Exception:
    traceback.print_exc()
    print("The summary failed (above); the data already on Drive is not affected.")

# Drive uploads lag behind the writes. Releasing the machine before they finish loses the newest files,
# so wait for them explicitly.
print("\nWaiting until every file has reached Google Drive (usually a few minutes) ...")
try:
    drive.flush_and_unmount(timeout_ms=40 * 60 * 1000)
    print("All files are on Google Drive.")
except Exception as exc:
    print(f"The upload check did not finish ({type(exc).__name__}: {exc}); the newest files may be missing.")

if RUN_STATE == "session_limit":
    print("\nColab's 12-hour session limit is close, so this session ends here. Later: open this notebook, "
          "Runtime -> Run all. Finished units are skipped; the budget is kept.")
elif RUN_STATE == "error":
    print("\nThe main runs stopped because of errors. Send Claude the output of cell 7.")
else:
    print("\nAll done. Download bb_traces_export.zip from MyDrive/bounded-brain/libero_runs/.")
if AUTO_DISCONNECT:
    from google.colab import runtime
    print("Releasing the GPU so no more compute units are used.")
    runtime.unassign()